# 10 · Model comparison & hyperparameter tuning

**Owner:** Teammate A
**Report section:** Model comparison & selection

**Goal.** Feature set × model grid (LogReg, Multinomial NB, LinearSVC + CalibratedClassifierCV, SGD log-loss, Random Forest, LightGBM) with a 5-fold CV AUROC table and bar chart, then a hyperparameter sweep for the top 2.

Run with *Restart & Run All*. Set `QUICK = True` for a fast debug pass on the 2,000-row
starter file, `QUICK = False` for the numbers that go in the report. Every result uses the
shared split from `src.data.get_split()` (seed 42), so numbers are comparable across notebooks.

**Outputs:** `results/model_comparison.csv`, `results/hparam_sweep.csv`, `results/preds_<best>.csv`, `figures/model_comparison.png`.

In [ ]:
# --- setup: make `src/` importable from anywhere (repo root, notebooks/, Kaggle/Colab clone) ---
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

QUICK = True  # True: 2,000-row starter file for fast debugging. False: full data (report numbers).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import SEED, set_seed, RESULTS_DIR, FIGURES_DIR
from src.data import get_split, load_test, load_unlabeled, load_train, get_cv_folds, xy
from src.features import build_features
from src.models import make_model
from src.evaluate import (auroc, cross_val_auroc, plot_roc, savefig, log_result,
                          save_predictions, predict_proba1)

set_seed(SEED)
SUFFIX = "_quick" if QUICK else ""   # keeps QUICK-mode outputs separate from real ones
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from src.features import FEATURE_SETS
try:
    from lightgbm import LGBMClassifier
except Exception as e:  # macOS: `brew install libomp`
    LGBMClassifier = None
    print("LightGBM unavailable:", e)

In [ ]:
train_df, val_df = get_split(quick=QUICK)
test_df = load_test()
X_tr, y_tr = xy(train_df)
X_va, y_va = xy(val_df)
full_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
# Candidate classifiers (fresh instance per call).
CLASSIFIERS = {
    "logreg":     lambda: LogisticRegression(C=1.0, max_iter=2000, solver="liblinear"),
    "mnb":        lambda: MultinomialNB(alpha=0.1),        # needs non-negative features (no scaled stylometric)
    "linsvc_cal": lambda: CalibratedClassifierCV(LinearSVC(C=0.5), cv=3),
    "sgd_log":    lambda: SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=50, random_state=SEED),
    "rf":         lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    "lgbm":       (lambda: LGBMClassifier(n_estimators=400, learning_rate=0.05, random_state=SEED, verbose=-1))
                  if LGBMClassifier else None,
}
print("feature sets:", FEATURE_SETS)

## TODO 1: feature set × model grid

* Loop over `FEATURE_SETS` × `CLASSIFIERS` and build
  `Pipeline([("features", build_features(fs)), ("clf", CLASSIFIERS[name]())])`.
* Score each with `cross_val_auroc(pipe, full_df["text"], full_df["label"])` and log every
  cell with `log_result("model_comparison", config, metrics)`. Also record fit time.
* Skip invalid combos: MNB needs non-negative input (`build_features("stylometric", scale=False)`
  is OK, but scaled stylometric is not). RF/LightGBM on ~10⁵ sparse TF-IDF columns is slow, so
  consider `max_features` (e.g. `word={"max_features": 20000}`) or TruncatedSVD.
* Start with `QUICK = True` to get the loop working.

In [ ]:
# TODO: grid loop goes here

## TODO 2: CV AUROC table and bar chart (`figures/model_comparison.png`, use `savefig`)

In [ ]:
# TODO

## TODO 3: hyperparameter sweep for the top 2 pipelines

e.g. LogReg `C`, TF-IDF `ngram_range` / `min_df` / `max_features`, and the char vs. char_wb analyzer.
Log to `results/hparam_sweep.csv`. For the winner, save predictions with
`save_predictions("<name>", val_df, val_probs, test_df, test_probs)` (val probs from the
train-split fit, test probs from a refit on `full_df`), and ask Allen to add it to `src/models.py`.

In [ ]:
# TODO